# Chapter 6 — Exercise Solutions## Policy Gradients, PPO & DPO[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 6.1: REINFORCE with Baseline

In [ ]:
import torch, torch.nn as nn, numpy as npclass PolicyNet(nn.Module):    def __init__(self, obs_dim=4, n_actions=2):        super().__init__()        self.net = nn.Sequential(            nn.Linear(obs_dim, 64), nn.Tanh(),            nn.Linear(64, 64),     nn.Tanh(),            nn.Linear(64, n_actions)        )    def forward(self, x):        return torch.distributions.Categorical(logits=self.net(x))def compute_returns(rewards, gamma=0.99):    G, returns = 0, []    for r in reversed(rewards):        G = r + gamma * G        returns.insert(0, G)    return torch.tensor(returns, dtype=torch.float32)def reinforce_with_baseline(env, n_episodes=500, lr=1e-3, gamma=0.99):    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)    ep_rewards = []    for ep in range(n_episodes):        obs, _ = env.reset()        log_probs, rewards, states = [], [], []        # Collect trajectory        for _ in range(500):            obs_t = torch.FloatTensor(obs)            dist = policy(obs_t)            action = dist.sample()            log_probs.append(dist.log_prob(action))            obs, r, terminated, truncated, _ = env.step(action.item())            rewards.append(r)            if terminated or truncated: break        ep_rewards.append(sum(rewards))        returns = compute_returns(rewards, gamma)        # KEY: subtract baseline (mean return) to reduce variance        baseline = returns.mean()                   # <-- the baseline        advantages = returns - baseline             # <-- advantages        # Normalise for stable gradients        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)        # Policy gradient loss with baseline        log_probs_t = torch.stack(log_probs)        loss = -(log_probs_t * advantages).mean()   # <-- advantage-weighted        optimizer.zero_grad()        loss.backward()        optimizer.step()    return ep_rewardsprint("REINFORCE + baseline: E[gradient] unchanged, Var[gradient] reduced")print("Baseline b can be any function of state (not action) — does NOT introduce bias")print("Mean return is simplest; V(s) (learned value function) is optimal baseline")# YOUR TURN: Compare training with baseline vs without on CartPole# try: import gymnasium; env = gymnasium.make('CartPole-v1')# rewards_no_baseline = reinforce_no_baseline(env)# rewards_with_baseline = reinforce_with_baseline(env)# Plot and compare variance of learning curves

### Solution 6.2: GAE with Different λ Values

In [ ]:
import torch, numpy as np, matplotlib.pyplot as pltdef compute_gae(rewards, values, next_values, dones, gamma=0.99, lam=0.95):    T = len(rewards)    advantages = [0.0] * T    gae = 0.0    for t in reversed(range(T)):        nv = 0.0 if dones[t] else next_values[t]        delta = rewards[t] + gamma * nv - values[t]        gae = delta + gamma * lam * (0.0 if dones[t] else gae)        advantages[t] = gae    adv = torch.FloatTensor(advantages)    return adv  # raw, not normalised (so we can compare)# Synthetic episode: sparse reward at endT = 30rewards     = [0.0] * 29 + [1.0]values      = [0.05] * Tnext_values = values[1:] + [0.0]dones       = [False] * 29 + [True]lambdas = [0.0, 0.5, 0.95, 1.0]fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)for ax, lam in zip(axes, lambdas):    adv = compute_gae(rewards, values, next_values, dones, lam=lam)    colors = ['steelblue' if a>=0 else 'tomato' for a in adv]    ax.bar(range(T), adv.numpy(), color=colors)    ax.set_title(f'λ = {lam}')    ax.set_xlabel('Step t')    ax.axhline(0, color='black', linewidth=0.5)    var = adv.var().item()    ax.set_ylabel(f'Advantage (Var={var:.4f})')plt.suptitle('GAE Credit Assignment: Effect of λ on Variance and Bias', fontsize=12)plt.tight_layout(); plt.show()for lam in lambdas:    adv = compute_gae(rewards, values, next_values, dones, lam=lam)    print(f"λ={lam:.2f}: var={adv.var():.4f}, max_adv={adv.max():.4f}, spread_steps={int((adv.abs()>0.01).sum())}")print("\nλ=0 (TD):  low variance, high bias — only adjacent step gets credit")print("λ=1 (MC):  high variance, no bias — credit spread across all steps")print("λ=0.95:   practical sweet spot used in PPO, GRPO, and most Actor-Critic")# YOUR TURN: Try a dense reward signal (r_t=0.1 for every step, +1 at end)# Does the choice of λ matter less for dense rewards?

### Solution 6.3: DPO Loss Sensitivity Analysis

In [ ]:
import torch, numpy as np, matplotlib.pyplot as pltdef dpo_loss_scalar(margin, beta):    """DPO loss for a scalar log-ratio margin."""    return -torch.nn.functional.logsigmoid(torch.tensor(beta * margin)).item()def dpo_gradient(margin, beta):    """Gradient of DPO loss w.r.t. margin."""    # d/d_margin [ -log σ(β*m) ] = -β * (1 - σ(β*m)) = -β * σ(-β*m)    return -beta * torch.sigmoid(torch.tensor(-beta * margin)).item()margins = np.linspace(-3, 3, 200)betas = [0.05, 0.1, 0.5]fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, beta in zip(axes, betas):    losses = [dpo_loss_scalar(m, beta) for m in margins]    grads  = [dpo_gradient(m, beta) for m in margins]    ax.plot(margins, losses, color='steelblue', linewidth=2, label='Loss')    ax2 = ax.twinx()    ax2.plot(margins, grads, color='tomato', linewidth=2, linestyle='--', label='Gradient')    ax.axvline(0, color='gray', linestyle=':')    ax.set_title(f'β = {beta}')    ax.set_xlabel('Log-ratio margin (chosen - rejected)')    ax.set_ylabel('DPO Loss', color='steelblue')    ax2.set_ylabel('Gradient', color='tomato')plt.suptitle('DPO Loss and Gradient: Effect of β', fontsize=12)plt.tight_layout(); plt.show()# Answer the three questionsprint("Q1: Gradient at zero margin:")for beta in betas:    print(f"  β={beta}: gradient = {dpo_gradient(0, beta):.4f}  (= -β/2)")print("\nQ2: At very large margins (>>0):")print("  Loss → 0, gradient → 0 (sigmoid saturates — like BCE saturation)")print("  High-confidence preferences no longer contribute gradients")print("\nQ3: Connection to binary cross-entropy saturation:")print("  DPO loss IS a form of BCE on the log-ratio difference")print("  The sigmoid saturation at large margins is identical to BCE vanishing gradients")print("  Solution: use IPO (squared loss) or add label smoothing to prevent saturation")# YOUR TURN: Plot gradient magnitude as function of margin for β ∈ {0.01, 5.0}# At what margin does the gradient drop below 1% of its peak value?